In [ ]:

#Code


# Install dependencies
!pip install transformers tensorflow nltk spacy
!python -m spacy download en_core_web_sm

# --- Import libraries ---
import numpy as np
import pandas as pd
import tensorflow as tf
import nltk
import spacy
import matplotlib.pyplot as plt

from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, Bidirectional, Dropout, Layer, TimeDistributed, Concatenate, Flatten, Activation
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
from transformers import BertTokenizer

nltk.download('punkt')
nlp = spacy.load("en_core_web_sm")

# --- Constants ---
MAX_SENTENCES = 15
MAX_WORDS = 20
BERT_VOCAB_SIZE = 30522

# --- Load ASAP 2.0 Dataset ---
df = pd.read_csv("asap2_dataset.csv")

# Clean text and fill missing
df['essay_text'] = df['essay_text'].fillna('').apply(lambda x: x.strip().replace('\n', ' '))
df['prompt_id'] = df['prompt_id'].astype(str).fillna('unknown')
df['gender'] = df['gender'].fillna('Unknown')
df['grade_level'] = df['grade_level'].fillna(df['grade_level'].median())

# Encode categorical variables
le_prompt = LabelEncoder()
df['prompt_id_enc'] = le_prompt.fit_transform(df['prompt_id'])

le_gender = LabelEncoder()
df['gender_enc'] = le_gender.fit_transform(df['gender'])

scaler = StandardScaler()
df['grade_scaled'] = scaler.fit_transform(df[['grade_level']])

# Normalize scores
df['score_norm'] = (df['score'] - df['score'].min()) / (df['score'].max() - df['score'].min())
df['grammar_norm'] = (df['grammar_score'] - df['grammar_score'].min()) / (df['grammar_score'].max() - df['grammar_score'].min()) if 'grammar_score' in df.columns else df['score_norm']

# --- Tokenizer and POS Vocab ---
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
pos_vocab = {'PAD': 0}
pos_counter = 1

def extract_syntax_semantics(essay):
    doc = nlp(essay)
    sentences = [sent.text for sent in doc.sents][:MAX_SENTENCES]
    tokenized_ids, pos_ids = [], []

    for sent in sentences:
        words = tokenizer.tokenize(sent)[:MAX_WORDS]
        word_ids = tokenizer.convert_tokens_to_ids(words)
        word_ids += [0] * (MAX_WORDS - len(word_ids))
        tokenized_ids.append(word_ids)

        pos_sent = [token.pos_ for token in nlp(sent)[:MAX_WORDS]]
        pos_id = []
        for pos in pos_sent:
            if pos not in pos_vocab:
                global pos_counter
                pos_vocab[pos] = pos_counter
                pos_counter += 1
            pos_id.append(pos_vocab[pos])
        pos_id += [0] * (MAX_WORDS - len(pos_id))
        pos_ids.append(pos_id)

    while len(tokenized_ids) < MAX_SENTENCES:
        tokenized_ids.append([0]*MAX_WORDS)
        pos_ids.append([0]*MAX_WORDS)

    return tokenized_ids, pos_ids

# --- Prepare Input Arrays ---
semantic_inputs, syntax_inputs = [], []
for essay in df['essay_text']:
    s, p = extract_syntax_semantics(essay)
    semantic_inputs.append(s)
    syntax_inputs.append(p)

X_sem = np.array(semantic_inputs)
X_syn = np.array(syntax_inputs)
X_meta = df[['gender_enc', 'grade_scaled']].values
Y_hol = df['score_norm'].values
Y_gram = df['grammar_norm'].values

# --- Train/Test Split ---
X_sem_train, X_sem_test, X_syn_train, X_syn_test, X_meta_train, X_meta_test, y_hol_train, y_hol_test, y_gram_train, y_gram_test = train_test_split(
    X_sem, X_syn, X_meta, Y_hol, Y_gram, test_size=0.2, random_state=42)

# --- Multi-head attention utility ---
def multi_head_attention(inputs, num_heads=4):
    dim = inputs.shape[-1]
    depth = dim // num_heads
    query = Dense(dim)(inputs)
    key = Dense(dim)(inputs)
    value = Dense(dim)(inputs)

    query = tf.reshape(query, (-1, inputs.shape[1], num_heads, depth))
    key = tf.reshape(key, (-1, inputs.shape[1], num_heads, depth))
    value = tf.reshape(value, (-1, inputs.shape[1], num_heads, depth))

    scores = tf.matmul(query, key, transpose_b=True) / tf.math.sqrt(tf.cast(depth, tf.float32))
    weights = tf.nn.softmax(scores, axis=-1)
    output = tf.matmul(weights, value)
    output = tf.reshape(output, (-1, inputs.shape[1], dim))
    return Dense(dim)(output)

# --- Build Novel Dual-Path Model ---
def build_novel_model():
    embed_dim = 128
    lstm_units = 64
    pos_vocab_size = len(pos_vocab)+1

    semantic_input = Input(shape=(MAX_SENTENCES, MAX_WORDS), dtype='int32', name='semantic_input')
    syntax_input = Input(shape=(MAX_SENTENCES, MAX_WORDS), dtype='int32', name='syntax_input')
    metadata_input = Input(shape=(2,), name='meta_input')

    word_input_sem = Input(shape=(MAX_WORDS,))
    word_emb_sem = Embedding(BERT_VOCAB_SIZE, embed_dim, mask_zero=True)(word_input_sem)
    word_lstm_sem = Bidirectional(LSTM(lstm_units, return_sequences=True))(word_emb_sem)
    word_sem_out = multi_head_attention(word_lstm_sem)
    word_sem_model = Model(word_input_sem, word_sem_out)

    word_input_syn = Input(shape=(MAX_WORDS,))
    word_emb_syn = Embedding(pos_vocab_size, embed_dim, mask_zero=True)(word_input_syn)
    word_lstm_syn = Bidirectional(LSTM(lstm_units, return_sequences=True))(word_emb_syn)
    word_syn_out = multi_head_attention(word_lstm_syn)
    word_syn_model = Model(word_input_syn, word_syn_out)

    encoded_sem = TimeDistributed(word_sem_model)(semantic_input)
    encoded_syn = TimeDistributed(word_syn_model)(syntax_input)

    sent_lstm_sem = Bidirectional(LSTM(lstm_units, return_sequences=True))(encoded_sem)
    sent_lstm_syn = Bidirectional(LSTM(lstm_units, return_sequences=True))(encoded_syn)

    gate = Dense(lstm_units*2, activation='sigmoid')(sent_lstm_sem)
    fused = gate * sent_lstm_sem + (1 - gate) * sent_lstm_syn

    attn_weights = Dense(1, activation='tanh')(fused)
    attn_weights = Flatten()(attn_weights)
    attn_weights = Activation('softmax')(attn_weights)
    attended = tf.reduce_sum(fused * tf.expand_dims(attn_weights, -1), axis=1)

    meta_dense = Dense(32, activation='relu')(metadata_input)
    meta_dense = Dropout(0.2)(meta_dense)

    combined = Concatenate()([attended, meta_dense])
    x = Dense(64, activation='relu')(combined)
    x = Dropout(0.3)(x)

    holistic_out = Dense(1, activation='sigmoid', name='holistic_output')(x)
    grammar_out = Dense(1, activation='sigmoid', name='grammar_output')(x)

    model = Model(inputs=[semantic_input, syntax_input, metadata_input], outputs=[holistic_out, grammar_out])
    model.compile(optimizer=Adam(1e-4),
                  loss={'holistic_output': 'mse', 'grammar_output': 'mse'},
                  metrics={'holistic_output': 'mae', 'grammar_output': 'mae'})
    return model

model = build_novel_model()
model.summary()

# --- Train ---
history = model.fit(
    x={'semantic_input': X_sem_train, 'syntax_input': X_syn_train, 'meta_input': X_meta_train},
    y={'holistic_output': y_hol_train, 'grammar_output': y_gram_train},
    validation_split=0.1,
    epochs=6,
    batch_size=16,
    verbose=1
)

# --- Evaluate ---
preds = model.predict({'semantic_input': X_sem_test, 'syntax_input': X_syn_test, 'meta_input': X_meta_test})
holistic_pred, grammar_pred = preds[0].flatten(), preds[1].flatten()

print("--- Holistic Score Evaluation ---")
print("MAE:", mean_absolute_error(y_hol_test, holistic_pred))
print("MSE:", mean_squared_error(y_hol_test, holistic_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_hol_test, holistic_pred)))
print("R^2:", r2_score(y_hol_test, holistic_pred))
print("Pearson r:", pearsonr(y_hol_test, holistic_pred)[0])
print("Spearman ρ:", spearmanr(y_hol_test, holistic_pred)[0])

print("\n--- Grammar Score Evaluation ---")
print("MAE:", mean_absolute_error(y_gram_test, grammar_pred))
print("MSE:", mean_squared_error(y_gram_test, grammar_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_gram_test, grammar_pred)))
print("R^2:", r2_score(y_gram_test, grammar_pred))
print("Pearson r:", pearsonr(y_gram_test, grammar_pred)[0])
print("Spearman ρ:", spearmanr(y_gram_test, grammar_pred)[0])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data (from your table)
epochs = np.arange(1, 21)

holistic_mae = np.array([0.183, 0.158, 0.142, 0.132, 0.124, 0.117, 0.111, 0.108, 0.104, 0.101,
                         0.099, 0.097, 0.096, 0.094, 0.092, 0.091, 0.090, 0.089, 0.089, 0.088])
grammar_mae = np.array([0.191, 0.166, 0.149, 0.137, 0.129, 0.122, 0.116, 0.112, 0.108, 0.105,
                       0.103, 0.101, 0.099, 0.097, 0.095, 0.094, 0.093, 0.092, 0.092, 0.091])

holistic_rmse = np.array([0.249, 0.219, 0.200, 0.187, 0.173, 0.164, 0.155, 0.149, 0.145, 0.141,
                          0.138, 0.134, 0.130, 0.127, 0.123, 0.120, 0.118, 0.117, 0.116, 0.115])
grammar_rmse = np.array([0.259, 0.230, 0.209, 0.197, 0.184, 0.173, 0.165, 0.158, 0.155, 0.148,
                        0.145, 0.141, 0.138, 0.134, 0.130, 0.126, 0.124, 0.122, 0.121, 0.120])

holistic_r2 = np.array([0.42, 0.55, 0.61, 0.66, 0.71, 0.74, 0.76, 0.78, 0.79, 0.80,
                       0.81, 0.82, 0.83, 0.84, 0.85, 0.86, 0.86, 0.87, 0.87, 0.88])
grammar_r2 = np.array([0.38, 0.50, 0.58, 0.63, 0.68, 0.71, 0.74, 0.76, 0.77, 0.79,
                     0.80, 0.81, 0.82, 0.83, 0.84, 0.85, 0.85, 0.86, 0.86, 0.87])

holistic_pearson = np.array([0.65, 0.72, 0.76, 0.78, 0.82, 0.84, 0.86, 0.87, 0.88, 0.89,
                             0.89, 0.90, 0.91, 0.91, 0.92, 0.92, 0.92, 0.92, 0.92, 0.92])
grammar_pearson = np.array([0.62, 0.70, 0.74, 0.76, 0.79, 0.81, 0.83, 0.85, 0.86, 0.87,
                           0.88, 0.89, 0.90, 0.91, 0.91, 0.92, 0.92, 0.92, 0.92, 0.92])

# Plotting
fig, axs = plt.subplots(2, 2, figsize=(15, 11))
#fig.suptitle('Model Performance over Epochs: Holistic vs Grammar', fontsize=18, fontweight='bold')

# Helper function to increase ticks font size
def set_ticks_fontsize(ax, size=12):
    ax.tick_params(axis='both', which='major', labelsize=size)

# 1. MAE
axs[0, 0].plot(epochs, holistic_mae, marker='o', label='Holistic MAE', color='tab:blue', linewidth=2)
axs[0, 0].plot(epochs, grammar_mae, marker='x', label='Grammar MAE', color='tab:orange', linewidth=2)
axs[0, 0].set_title('MAE over Epochs (Lower is Better)', fontsize=14, fontweight='bold')
axs[0, 0].set_xlabel('Epoch', fontsize=12)
axs[0, 0].set_ylabel('MAE', fontsize=12)
axs[0, 0].invert_yaxis()  # Lower MAE is better
axs[0, 0].grid(True, linestyle='--', alpha=0.6)
axs[0, 0].legend(fontsize=11)
set_ticks_fontsize(axs[0, 0])

# Highlight critical region: steady improvement after epoch 5
axs[0, 0].axvspan(5, 20, color='lightgreen', alpha=0.3)
axs[0, 0].annotate('Consistent error reduction', xy=(15, 0.095), xytext=(8, 0.16),
                   arrowprops=dict(facecolor='green', arrowstyle='->', lw=2),
                   fontsize=12, fontweight='bold', color='darkgreen')

# Annotate significant values for MAE
for epoch in [1, 5, 10, 15, 20]:
    axs[0, 0].annotate(f"{holistic_mae[epoch-1]:.3f}", (epoch, holistic_mae[epoch-1]),
                       textcoords="offset points", xytext=(0,10), ha='center', fontsize=12, color='tab:blue')
    axs[0, 0].annotate(f"{grammar_mae[epoch-1]:.3f}", (epoch, grammar_mae[epoch-1]),
                       textcoords="offset points", xytext=(0,-15), ha='center', fontsize=12, color='tab:orange')

# 2. RMSE
axs[0, 1].plot(epochs, holistic_rmse, marker='o', label='Holistic RMSE', color='tab:blue', linewidth=2)
axs[0, 1].plot(epochs, grammar_rmse, marker='x', label='Grammar RMSE', color='tab:orange', linewidth=2)
axs[0, 1].set_title('RMSE over Epochs (Lower is Better)', fontsize=14, fontweight='bold')
axs[0, 1].set_xlabel('Epoch', fontsize=12)
axs[0, 1].set_ylabel('RMSE', fontsize=12)
axs[0, 1].invert_yaxis()
axs[0, 1].grid(True, linestyle='--', alpha=0.6)
axs[0, 1].legend(fontsize=11)
set_ticks_fontsize(axs[0, 1])

# Highlight critical region: rapid error drop first 7 epochs
axs[0, 1].axvspan(1, 7, color='yellow', alpha=0.3)
# Shortened arrow length here
axs[0, 1].annotate('Rapid RMSE decrease', xy=(6, 0.17), xytext=(6, 0.21),
                   arrowprops=dict(facecolor='orange', arrowstyle='->', lw=2),
                   fontsize=12, fontweight='bold', color='darkorange')

# Annotate significant values for RMSE
for epoch in [1, 5, 10, 15, 20]:
    axs[0, 1].annotate(f"{holistic_rmse[epoch-1]:.3f}", (epoch, holistic_rmse[epoch-1]),
                       textcoords="offset points", xytext=(0,10), ha='center', fontsize=12, color='tab:blue')
    axs[0, 1].annotate(f"{grammar_rmse[epoch-1]:.3f}", (epoch, grammar_rmse[epoch-1]),
                       textcoords="offset points", xytext=(0,-15), ha='center', fontsize=12, color='tab:orange')

# 3. R²
axs[1, 0].plot(epochs, holistic_r2, marker='o', label='Holistic R²', color='tab:blue', linewidth=2)
axs[1, 0].plot(epochs, grammar_r2, marker='x', label='Grammar R²', color='tab:orange', linewidth=2)
axs[1, 0].set_title('R² over Epochs (Higher is Better)', fontsize=14, fontweight='bold')
axs[1, 0].set_xlabel('Epoch', fontsize=12)
axs[1, 0].set_ylabel('R²', fontsize=12)
axs[1, 0].set_ylim(0.35, 0.9)
axs[1, 0].grid(True, linestyle='--', alpha=0.6)
axs[1, 0].legend(fontsize=11)
set_ticks_fontsize(axs[1, 0])

# Highlight critical region: solid high accuracy after epoch 12
axs[1, 0].axvspan(12, 20, color='lightblue', alpha=0.3)
axs[1, 0].annotate('Strong explanatory power', xy=(15, 0.83), xytext=(7, 0.55),
                   arrowprops=dict(facecolor='blue', arrowstyle='->', lw=2),
                   fontsize=12, fontweight='bold', color='navy')

# Annotate significant values for R²
for epoch in [1, 5, 10, 15, 20]:
    axs[1, 0].annotate(f"{holistic_r2[epoch-1]:.2f}", (epoch, holistic_r2[epoch-1]),
                       textcoords="offset points", xytext=(0,10), ha='center', fontsize=12, color='tab:blue')
    axs[1, 0].annotate(f"{grammar_r2[epoch-1]:.2f}", (epoch, grammar_r2[epoch-1]),
                       textcoords="offset points", xytext=(0,-15), ha='center', fontsize=12, color='tab:orange')

# 4. Pearson r
axs[1, 1].plot(epochs, holistic_pearson, marker='o', label='Holistic Pearson r', color='tab:blue', linewidth=2)
axs[1, 1].plot(epochs, grammar_pearson, marker='x', label='Grammar Pearson r', color='tab:orange', linewidth=2)
axs[1, 1].set_title('Pearson Correlation over Epochs (Higher is Better)', fontsize=14, fontweight='bold')
axs[1, 1].set_xlabel('Epoch', fontsize=12)
axs[1, 1].set_ylabel('Pearson r', fontsize=12)
axs[1, 1].set_ylim(0.5, 1)
axs[1, 1].grid(True, linestyle='--', alpha=0.6)
axs[1, 1].legend(fontsize=11)
set_ticks_fontsize(axs[1, 1])

# Highlight critical region: convergence >0.9 from epoch 13 onwards
axs[1, 1].axvspan(13, 20, color='plum', alpha=0.3)
axs[1, 1].annotate('High correlation convergence', xy=(16, 0.92), xytext=(8, 0.7),
                   arrowprops=dict(facecolor='purple', arrowstyle='->', lw=2),
                   fontsize=12, fontweight='bold', color='purple')

# Annotate significant values for Pearson r
for epoch in [1, 5, 10, 15, 20]:
    axs[1, 1].annotate(f"{holistic_pearson[epoch-1]:.2f}", (epoch, holistic_pearson[epoch-1]),
                       textcoords="offset points", xytext=(0,10), ha='center', fontsize=12, color='tab:blue')
    axs[1, 1].annotate(f"{grammar_pearson[epoch-1]:.2f}", (epoch, grammar_pearson[epoch-1]),
                       textcoords="offset points", xytext=(0,-15), ha='center', fontsize=12, color='tab:orange')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data for 20th epoch (SYNSEMNet) and approximate base study (RoBERTa+BiLSTM)
epochs = np.arange(1, 21)

# SYNSEMNet final epoch metrics (constant for visualization simplicity)
syn_mae_holistic = 0.088
syn_pearson_holistic = 0.92
syn_mae_grammar = 0.091
syn_pearson_grammar = 0.92

# RoBERTa+BiLSTM approximate values (simulate flat performance around reported ranges)
base_mae_holistic = 0.11
base_pearson_holistic = 0.90
base_mae_grammar = 0.12
base_pearson_grammar = 0.89

# Create figure and axes
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Performance Comparison: SYNSEMNet vs RoBERTa + BiLSTM AES Model", fontsize=20, weight='bold')

# Custom bar properties
bar_props = {
    'SYNSEMNet': {'color': '#2ca02c', 'alpha': 0.85, 'edgecolor': 'darkgreen', 'hatch': '///'},
    'RoBERTa+BiLSTM': {'color': '#d62728', 'alpha': 0.85, 'edgecolor': 'darkred', 'hatch': '\\\\\\'}
}

# --- Plot 1: Holistic MAE (lower is better) ---
ax = axs[0, 0]
bars = ax.bar(['RoBERTa+BiLSTM', 'SYNSEMNet'], [base_mae_holistic, syn_mae_holistic],
              **bar_props['RoBERTa+BiLSTM'])
bars[1].set_color(bar_props['SYNSEMNet']['color'])
bars[1].set_alpha(bar_props['SYNSEMNet']['alpha'])
bars[1].set_edgecolor(bar_props['SYNSEMNet']['edgecolor'])
bars[1].set_hatch(bar_props['SYNSEMNet']['hatch'])
bars[0].set_edgecolor(bar_props['RoBERTa+BiLSTM']['edgecolor'])
bars[0].set_hatch(bar_props['RoBERTa+BiLSTM']['hatch'])

ax.set_title("Holistic MAE (Lower is Better)", fontsize=16, weight='bold')
ax.set_ylabel("Mean Absolute Error", fontsize=14)
ax.tick_params(axis='both', labelsize=13)
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Highlight critical region with hatch
ax.axhspan(0, 0.12, color='green', alpha=0.12, hatch='..')
ax.text(0.5, 0.11, "Better Accuracy Region", color='green', fontsize=12, ha='center', weight='bold')

# Arrow annotation
ax.annotate('Lower MAE → Better', xy=(1, syn_mae_holistic), xytext=(0.7, 0.15),
            arrowprops=dict(facecolor='green', arrowstyle='->', lw=2, alpha=0.9), fontsize=13, color='green', weight='bold')
ax.annotate(f"{syn_mae_holistic:.3f}", xy=(1, syn_mae_holistic), xytext=(1.1, syn_mae_holistic),
            fontsize=13, color='green', weight='bold')
ax.annotate(f"{base_mae_holistic:.3f}", xy=(0, base_mae_holistic), xytext=(-0.15, base_mae_holistic),
            fontsize=13, color='red', weight='bold')

# --- Plot 2: Holistic Pearson r (higher is better) ---
ax = axs[0, 1]
bars = ax.bar(['RoBERTa+BiLSTM', 'SYNSEMNet'], [base_pearson_holistic, syn_pearson_holistic],
              **bar_props['RoBERTa+BiLSTM'])
bars[1].set_color(bar_props['SYNSEMNet']['color'])
bars[1].set_alpha(bar_props['SYNSEMNet']['alpha'])
bars[1].set_edgecolor(bar_props['SYNSEMNet']['edgecolor'])
bars[1].set_hatch(bar_props['SYNSEMNet']['hatch'])
bars[0].set_edgecolor(bar_props['RoBERTa+BiLSTM']['edgecolor'])
bars[0].set_hatch(bar_props['RoBERTa+BiLSTM']['hatch'])

ax.set_title("Holistic Pearson Correlation (Higher is Better)", fontsize=16, weight='bold')
ax.set_ylabel("Pearson r", fontsize=14)
ax.set_ylim(0.85, 0.95)
ax.tick_params(axis='both', labelsize=13)
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Highlight critical region with hatch
ax.axhspan(0.90, 0.95, color='green', alpha=0.12, hatch='..')
ax.text(0.5, 0.91, "High Correlation Zone", color='green', fontsize=12, ha='center', weight='bold')

# Arrow annotation
ax.annotate('Higher r → Better', xy=(1, syn_pearson_holistic), xytext=(0.7, 0.88),
            arrowprops=dict(facecolor='green', arrowstyle='->', lw=2, alpha=0.9), fontsize=13, color='green', weight='bold')
ax.annotate(f"{syn_pearson_holistic:.2f}", xy=(1, syn_pearson_holistic), xytext=(1.1, syn_pearson_holistic),
            fontsize=13, color='green', weight='bold')
ax.annotate(f"{base_pearson_holistic:.2f}", xy=(0, base_pearson_holistic), xytext=(-0.15, base_pearson_holistic),
            fontsize=13, color='red', weight='bold')

# --- Plot 3: Grammar MAE (lower is better) ---
ax = axs[1, 0]
bars = ax.bar(['RoBERTa+BiLSTM', 'SYNSEMNet'], [base_mae_grammar, syn_mae_grammar],
              **bar_props['RoBERTa+BiLSTM'])
bars[1].set_color(bar_props['SYNSEMNet']['color'])
bars[1].set_alpha(bar_props['SYNSEMNet']['alpha'])
bars[1].set_edgecolor(bar_props['SYNSEMNet']['edgecolor'])
bars[1].set_hatch(bar_props['SYNSEMNet']['hatch'])
bars[0].set_edgecolor(bar_props['RoBERTa+BiLSTM']['edgecolor'])
bars[0].set_hatch(bar_props['RoBERTa+BiLSTM']['hatch'])

ax.set_title("Grammar MAE (Lower is Better)", fontsize=16, weight='bold')
ax.set_ylabel("Mean Absolute Error", fontsize=14)
ax.tick_params(axis='both', labelsize=13)
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Highlight critical region with hatch
ax.axhspan(0, 0.13, color='green', alpha=0.12, hatch='..')
ax.text(0.5, 0.12, "Better Accuracy Region", color='green', fontsize=12, ha='center', weight='bold')

# Arrow annotation
ax.annotate('Lower MAE → Better', xy=(1, syn_mae_grammar), xytext=(0.7, 0.16),
            arrowprops=dict(facecolor='green', arrowstyle='->', lw=2, alpha=0.9), fontsize=13, color='green', weight='bold')
ax.annotate(f"{syn_mae_grammar:.3f}", xy=(1, syn_mae_grammar), xytext=(1.1, syn_mae_grammar),
            fontsize=13, color='green', weight='bold')
ax.annotate(f"{base_mae_grammar:.3f}", xy=(0, base_mae_grammar), xytext=(-0.15, base_mae_grammar),
            fontsize=13, color='red', weight='bold')

# --- Plot 4: Grammar Pearson r (higher is better) ---
ax = axs[1, 1]
bars = ax.bar(['RoBERTa+BiLSTM', 'SYNSEMNet'], [base_pearson_grammar, syn_pearson_grammar],
              **bar_props['RoBERTa+BiLSTM'])
bars[1].set_color(bar_props['SYNSEMNet']['color'])
bars[1].set_alpha(bar_props['SYNSEMNet']['alpha'])
bars[1].set_edgecolor(bar_props['SYNSEMNet']['edgecolor'])
bars[1].set_hatch(bar_props['SYNSEMNet']['hatch'])
bars[0].set_edgecolor(bar_props['RoBERTa+BiLSTM']['edgecolor'])
bars[0].set_hatch(bar_props['RoBERTa+BiLSTM']['hatch'])

ax.set_title("Grammar Pearson Correlation (Higher is Better)", fontsize=16, weight='bold')
ax.set_ylabel("Pearson r", fontsize=14)
ax.set_ylim(0.85, 0.95)
ax.tick_params(axis='both', labelsize=13)
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Highlight critical region with hatch
ax.axhspan(0.90, 0.95, color='green', alpha=0.12, hatch='..')
ax.text(0.5, 0.91, "High Correlation Zone", color='green', fontsize=12, ha='center', weight='bold')

# Arrow annotation
ax.annotate('Higher r → Better', xy=(1, syn_pearson_grammar), xytext=(0.7, 0.88),
            arrowprops=dict(facecolor='green', arrowstyle='->', lw=2, alpha=0.9), fontsize=13, color='green', weight='bold')
ax.annotate(f"{syn_pearson_grammar:.2f}", xy=(1, syn_pearson_grammar), xytext=(1.1, syn_pearson_grammar),
            fontsize=13, color='green', weight='bold')
ax.annotate(f"{base_pearson_grammar:.2f}", xy=(0, base_pearson_grammar), xytext=(-0.15, base_pearson_grammar),
            fontsize=13, color='red', weight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
# Install packages for model plotting (run once in Colab)
!apt-get install -y graphviz
!pip install pydot

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Embedding, LSTM, Bidirectional, Dropout,
                                     TimeDistributed, Concatenate, Flatten, Activation,
                                     Layer, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import plot_model
from IPython.display import Image

# Constants
MAX_SENTENCES = 15
MAX_WORDS = 20
BERT_VOCAB_SIZE = 30522

# Example pos_vocab (extend as needed)
pos_vocab = {'PAD': 0, 'NOUN': 1, 'VERB': 2}

# Custom MultiHeadAttention Layer
class MultiHeadAttention(Layer):
    def __init__(self, num_heads=4, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads

    def build(self, input_shape):
        dim = input_shape[-1]
        self.dim = dim
        self.depth = dim // self.num_heads

        self.wq = Dense(dim)
        self.wk = Dense(dim)
        self.wv = Dense(dim)
        self.dense = Dense(dim)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]

        query = self.wq(inputs)
        key = self.wk(inputs)
        value = self.wv(inputs)

        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)

        scores = tf.matmul(query, key, transpose_b=True) / tf.math.sqrt(tf.cast(self.depth, tf.float32))
        weights = tf.nn.softmax(scores, axis=-1)
        attention_output = tf.matmul(weights, value)

        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention_output, (batch_size, -1, self.dim))

        output = self.dense(concat_attention)
        return output

# Custom Layer to do weighted sum attention safely inside model
class AttentionWeightedSum(Layer):
    def __init__(self, **kwargs):
        super(AttentionWeightedSum, self).__init__(**kwargs)

    def call(self, inputs):
        fused, attn_weights = inputs  # unpack inputs
        attn_weights_exp = tf.expand_dims(attn_weights, axis=-1)  # expand dims for broadcast
        weighted_sum = tf.reduce_sum(fused * attn_weights_exp, axis=1)  # sum over time dimension
        return weighted_sum

# Build model function
def build_novel_model():
    embed_dim = 128
    lstm_units = 64
    pos_vocab_size = len(pos_vocab) + 1

    semantic_input = Input(shape=(MAX_SENTENCES, MAX_WORDS), dtype='int32', name='semantic_input')
    syntax_input = Input(shape=(MAX_SENTENCES, MAX_WORDS), dtype='int32', name='syntax_input')
    metadata_input = Input(shape=(2,), name='meta_input')

    # Word-level semantic submodel
    word_input_sem = Input(shape=(MAX_WORDS,))
    word_emb_sem = Embedding(BERT_VOCAB_SIZE, embed_dim, mask_zero=True)(word_input_sem)
    word_lstm_sem = Bidirectional(LSTM(lstm_units, return_sequences=True))(word_emb_sem)
    word_sem_out = MultiHeadAttention(num_heads=4)(word_lstm_sem)
    word_sem_out = GlobalAveragePooling1D()(word_sem_out)
    word_sem_model = Model(word_input_sem, word_sem_out)

    # Word-level syntax submodel
    word_input_syn = Input(shape=(MAX_WORDS,))
    word_emb_syn = Embedding(pos_vocab_size, embed_dim, mask_zero=True)(word_input_syn)
    word_lstm_syn = Bidirectional(LSTM(lstm_units, return_sequences=True))(word_emb_syn)
    word_syn_out = MultiHeadAttention(num_heads=4)(word_lstm_syn)
    word_syn_out = GlobalAveragePooling1D()(word_syn_out)
    word_syn_model = Model(word_input_syn, word_syn_out)

    # Sentence-level encodings
    encoded_sem = TimeDistributed(word_sem_model)(semantic_input)  # shape (batch, 15, 128)
    encoded_syn = TimeDistributed(word_syn_model)(syntax_input)    # shape (batch, 15, 128)

    sent_lstm_sem = Bidirectional(LSTM(lstm_units, return_sequences=True))(encoded_sem)
    sent_lstm_syn = Bidirectional(LSTM(lstm_units, return_sequences=True))(encoded_syn)

    gate = Dense(lstm_units * 2, activation='sigmoid')(sent_lstm_sem)
    fused = gate * sent_lstm_sem + (1 - gate) * sent_lstm_syn

    attn_weights = Dense(1, activation='tanh')(fused)
    attn_weights = Flatten()(attn_weights)
    attn_weights = Activation('softmax')(attn_weights)

    # Use custom layer for weighted sum attention
    attended = AttentionWeightedSum()([fused, attn_weights])

    meta_dense = Dense(32, activation='relu')(metadata_input)
    meta_dense = Dropout(0.2)(meta_dense)

    combined = Concatenate()([attended, meta_dense])
    x = Dense(64, activation='relu')(combined)
    x = Dropout(0.3)(x)

    holistic_out = Dense(1, activation='sigmoid', name='holistic_output')(x)
    grammar_out = Dense(1, activation='sigmoid', name='grammar_output')(x)

    model = Model(inputs=[semantic_input, syntax_input, metadata_input], outputs=[holistic_out, grammar_out])

    model.compile(optimizer=Adam(1e-4),
                  loss={'holistic_output': 'mse', 'grammar_output': 'mse'},
                  metrics={'holistic_output': 'mae', 'grammar_output': 'mae'})

    return model

# Build model
model = build_novel_model()

# Print summary
model.summary()

# Plot and save architecture
plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True)

# Display architecture inline
Image('model_architecture.png')


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Embedding, LSTM, Bidirectional, Dropout,
                                     TimeDistributed, Concatenate, Flatten, Activation,
                                     Layer, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import seaborn as sns

MAX_SENTENCES = 15
MAX_WORDS = 20
BERT_VOCAB_SIZE = 30522
pos_vocab = {'PAD': 0, 'NOUN': 1, 'VERB': 2}  # example pos_vocab size

class MultiHeadAttention(Layer):
    def __init__(self, num_heads=4, output_attentions=False, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.output_attentions = output_attentions

    def build(self, input_shape):
        dim = input_shape[-1]
        self.dim = dim
        self.depth = dim // self.num_heads
        self.wq = Dense(dim)
        self.wk = Dense(dim)
        self.wv = Dense(dim)
        self.dense = Dense(dim)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        query = self.wq(inputs)
        key = self.wk(inputs)
        value = self.wv(inputs)

        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)

        scores = tf.matmul(query, key, transpose_b=True) / tf.math.sqrt(tf.cast(self.depth, tf.float32))
        weights = tf.nn.softmax(scores, axis=-1)
        attention_output = tf.matmul(weights, value)

        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention_output, (batch_size, -1, self.dim))
        output = self.dense(concat_attention)

        if self.output_attentions:
            return output, weights
        else:
            return output

class AttentionWeightedSum(Layer):
    def call(self, inputs):
        fused, attn_weights = inputs
        attn_weights_exp = tf.expand_dims(attn_weights, axis=-1)
        weighted_sum = tf.reduce_sum(fused * attn_weights_exp, axis=1)
        return weighted_sum

def build_training_model():
    embed_dim = 128
    lstm_units = 64
    pos_vocab_size = len(pos_vocab) + 1

    semantic_input = Input(shape=(MAX_SENTENCES, MAX_WORDS), dtype='int32', name='semantic_input')
    syntax_input = Input(shape=(MAX_SENTENCES, MAX_WORDS), dtype='int32', name='syntax_input')
    metadata_input = Input(shape=(2,), name='meta_input')

    # Word-level semantic submodel without output_attentions
    word_input_sem = Input(shape=(MAX_WORDS,))
    word_emb_sem = Embedding(BERT_VOCAB_SIZE, embed_dim, mask_zero=True)(word_input_sem)
    word_lstm_sem = Bidirectional(LSTM(lstm_units, return_sequences=True))(word_emb_sem)
    word_sem_out = MultiHeadAttention(num_heads=4)(word_lstm_sem)
    word_sem_out = GlobalAveragePooling1D()(word_sem_out)
    word_sem_model = Model(word_input_sem, word_sem_out)

    # Word-level syntax submodel
    word_input_syn = Input(shape=(MAX_WORDS,))
    word_emb_syn = Embedding(pos_vocab_size, embed_dim, mask_zero=True)(word_input_syn)
    word_lstm_syn = Bidirectional(LSTM(lstm_units, return_sequences=True))(word_emb_syn)
    word_syn_out = MultiHeadAttention(num_heads=4)(word_lstm_syn)
    word_syn_out = GlobalAveragePooling1D()(word_syn_out)
    word_syn_model = Model(word_input_syn, word_syn_out)

    encoded_sem = TimeDistributed(word_sem_model)(semantic_input)
    encoded_syn = TimeDistributed(word_syn_model)(syntax_input)

    sent_lstm_sem = Bidirectional(LSTM(lstm_units, return_sequences=True))(encoded_sem)
    sent_lstm_syn = Bidirectional(LSTM(lstm_units, return_sequences=True))(encoded_syn)

    gate = Dense(lstm_units * 2, activation='sigmoid')(sent_lstm_sem)
    fused = gate * sent_lstm_sem + (1 - gate) * sent_lstm_syn

    attn_weights = Dense(1, activation='tanh')(fused)
    attn_weights = Flatten()(attn_weights)
    attn_weights = Activation('softmax')(attn_weights)
    attended = AttentionWeightedSum()([fused, attn_weights])

    meta_dense = Dense(32, activation='relu')(metadata_input)
    meta_dense = Dropout(0.2)(meta_dense)

    combined = Concatenate()([attended, meta_dense])
    x = Dense(64, activation='relu')(combined)
    x = Dropout(0.3)(x)

    holistic_out = Dense(1, activation='sigmoid', name='holistic_output')(x)
    grammar_out = Dense(1, activation='sigmoid', name='grammar_output')(x)

    model = Model(inputs=[semantic_input, syntax_input, metadata_input], outputs=[holistic_out, grammar_out])
    model.compile(optimizer=Adam(1e-4), loss='mse', metrics=['mae'])
    return model

def build_attention_extraction_model():
    embed_dim = 128
    lstm_units = 64

    # Single sentence input (no TimeDistributed)
    word_input_sem = Input(shape=(MAX_WORDS,))
    word_emb_sem = Embedding(BERT_VOCAB_SIZE, embed_dim, mask_zero=True)(word_input_sem)
    word_lstm_sem = Bidirectional(LSTM(lstm_units, return_sequences=True))(word_emb_sem)
    word_sem_out, attn_weights = MultiHeadAttention(num_heads=4, output_attentions=True)(word_lstm_sem)

    model = Model(inputs=word_input_sem, outputs=[word_sem_out, attn_weights])
    return model

# Build the training model
training_model = build_training_model()
training_model.summary()
training_model.save('novel_model.h5')  # Save for Netron visualization

# Build the attention extraction model for visualization
attention_model = build_attention_extraction_model()
attention_model.summary()

# Generate dummy input data (batch size 1)
dummy_input = np.random.randint(0, BERT_VOCAB_SIZE, (1, MAX_WORDS))

# Get output and attention weights from attention model
semantic_out, attn_weights = attention_model.predict(dummy_input)

print("Attention weights shape:", attn_weights.shape)  # (batch, heads, seq_len, seq_len)

# Plot first attention head heatmap
sns.heatmap(attn_weights[0][0], cmap='viridis')
plt.title("Multi-Head Attention Heatmap (Head 0)")
plt.xlabel("Key Positions")
plt.ylabel("Query Positions")
plt.show()
